In [1]:
from pathlib import Path
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid", palette="viridis")

DATA_DIR = Path.cwd()
if not (DATA_DIR / "hackathon_preli_dataset_-_bangladesh_dengue_large_dataset.csv").exists():
    DATA_DIR = DATA_DIR / "Trial"

RAW_PATH = DATA_DIR / "hackathon_preli_dataset_-_bangladesh_dengue_large_dataset.csv"
PROCESSED_PATH = DATA_DIR / "bangladesh_dengue_preprocessed.csv"
MODELS_DIR = DATA_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_csv(RAW_PATH)
print(f"Loaded raw dataset: {RAW_PATH.name} -> {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns")
print("Missing values:", int(raw_df.isna().sum().sum()))
print("Exact duplicates:", int(raw_df.duplicated().sum()))
display(raw_df.head(5))

Loaded raw dataset: hackathon_preli_dataset_-_bangladesh_dengue_large_dataset.csv -> 31,320 rows x 12 columns
Missing values: 0
Exact duplicates: 0


,date,city,ward_id,population_density_sqkm,water_logging_risk_index,sanitation_score,avg_temp_c,max_temp_c,min_temp_c,rainfall_mm,relative_humidity_pct,dengue_cases
0,2016-01-04,Chattogram,CCC_Ward_01,64024,0.87,0.46,18.32,26.54,14.13,0.00,59.24,0
1,2016-01-11,Chattogram,CCC_Ward_01,64024,0.87,0.46,21.24,26.95,16.71,2.02,61.63,0
2,2016-01-18,Chattogram,CCC_Ward_01,64024,0.87,0.46,20.94,25.05,14.84,0.00,60.50,0
3,2016-01-25,Chattogram,CCC_Ward_01,64024,0.87,0.46,21.75,27.20,17.58,1.52,69.02,0
4,2016-02-01,Chattogram,CCC_Ward_01,64024,0.87,0.46,22.39,28.02,16.98,0.94,60.50,0


In [2]:
def build_multiweek_lag_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Programmatically compute grouped multi-week weather lags, rolling rainfall sums,
    calendar features, city codes, and 2-to-4-week early-warning targets per ward_id."""
    work = df.copy()
    work["city"] = work["city"].astype(str).str.strip()
    work["ward_id"] = work["ward_id"].astype(str).str.strip()
    work["date"] = pd.to_datetime(work["date"], errors="raise")
    work = work.drop_duplicates(subset=["city", "ward_id", "date"])
    work = work.sort_values(["ward_id", "date"]).reset_index(drop=True)

    grp = work.groupby("ward_id", group_keys=False)

    for lag in range(1, 7):
        work[f"rain_lag{lag}"] = grp["rainfall_mm"].shift(lag)

    for lag in range(1, 9):
        work[f"temp_lag{lag}"] = grp["avg_temp_c"].shift(lag)
        work[f"hum_lag{lag}"] = grp["relative_humidity_pct"].shift(lag)

    work["rain_sum2w"] = work["rainfall_mm"] + work["rain_lag1"]
    work["rain_sum3w"] = work["rain_sum2w"] + work["rain_lag2"]
    work["rain_sum4w"] = work["rain_sum3w"] + work["rain_lag3"]

    work["cases_lag1"] = grp["dengue_cases"].shift(1)
    work["cases_lag2"] = grp["dengue_cases"].shift(2)

    work["target_2w"] = grp["dengue_cases"].shift(-2)
    work["target_3w"] = grp["dengue_cases"].shift(-3)
    work["target_4w"] = grp["dengue_cases"].shift(-4)

    work["week_of_year"] = work["date"].dt.isocalendar().week.astype(int)
    work["month"] = work["date"].dt.month
    city_map = {c: idx for idx, c in enumerate(sorted(work["city"].unique()))}
    work["city_code"] = work["city"].map(city_map).astype(int)

    work = work.dropna().reset_index(drop=True)
    return work


lag_df = build_multiweek_lag_dataset(raw_df)
print(f"Lag pipeline output shape: {lag_df.shape[0]:,} rows x {lag_df.shape[1]} columns")
print(f"Date range: {lag_df['date'].min().date()} to {lag_df['date'].max().date()} ({lag_df['ward_id'].nunique()} wards)")
print(f"Total missing values: {int(lag_df.isna().sum().sum())}")
display(lag_df.head(5))

Lag pipeline output shape: 30,600 rows x 45 columns
Date range: 2016-02-29 to 2025-12-01 (60 wards)
Total missing values: 0


,date,city,ward_id,population_density_sqkm,water_logging_risk_index,sanitation_score,avg_temp_c,max_temp_c,min_temp_c,rainfall_mm,relative_humidity_pct,dengue_cases,rain_lag1,rain_lag2,rain_lag3,rain_lag4,rain_lag5,rain_lag6,temp_lag1,hum_lag1,temp_lag2,hum_lag2,temp_lag3,hum_lag3,temp_lag4,hum_lag4,temp_lag5,hum_lag5,temp_lag6,hum_lag6,temp_lag7,hum_lag7,temp_lag8,hum_lag8,rain_sum2w,rain_sum3w,rain_sum4w,cases_lag1,cases_lag2,target_2w,target_3w,target_4w,week_of_year,month,city_code
0,2016-02-29,Chattogram,CCC_Ward_01,64024,0.87,0.46,25.22,29.47,19.23,5.89,55.06,0,3.61,8.19,4.17,0.94,1.52,0.00,22.66,60.54,23.56,52.70,24.55,49.88,22.39,60.50,21.75,69.02,20.94,60.50,21.24,61.63,18.32,59.24,9.50,17.69,21.86,0.0,0.0,0.0,0.0,0.0,9,2,0
1,2016-03-07,Chattogram,CCC_Ward_01,64024,0.87,0.46,25.71,31.31,21.56,0.00,64.79,0,5.89,3.61,8.19,4.17,0.94,1.52,25.22,55.06,22.66,60.54,23.56,52.70,24.55,49.88,22.39,60.50,21.75,69.02,20.94,60.50,21.24,61.63,5.89,9.50,17.69,0.0,0.0,0.0,0.0,0.0,10,3,0
2,2016-03-14,Chattogram,CCC_Ward_01,64024,0.87,0.46,25.80,32.82,21.54,2.68,62.29,0,0.00,5.89,3.61,8.19,4.17,0.94,25.71,64.79,25.22,55.06,22.66,60.54,23.56,52.70,24.55,49.88,22.39,60.50,21.75,69.02,20.94,60.50,2.68,8.57,12.18,0.0,0.0,0.0,0.0,0.0,11,3,0
3,2016-03-21,Chattogram,CCC_Ward_01,64024,0.87,0.46,26.37,30.18,19.64,0.00,62.55,0,2.68,0.00,5.89,3.61,8.19,4.17,25.80,62.29,25.71,64.79,25.22,55.06,22.66,60.54,23.56,52.70,24.55,49.88,22.39,60.50,21.75,69.02,2.68,2.68,8.57,0.0,0.0,0.0,0.0,0.0,12,3,0
4,2016-03-28,Chattogram,CCC_Ward_01,64024,0.87,0.46,26.01,31.69,21.28,0.00,66.45,0,0.00,2.68,0.00,5.89,3.61,8.19,26.37,62.55,25.80,62.29,25.71,64.79,25.22,55.06,22.66,60.54,23.56,52.70,24.55,49.88,22.39,60.50,0.00,2.68,2.68,0.0,0.0,0.0,0.0,0.0,13,3,0
